In [0]:
# ==================================================
# 1. CONFIGURATION
# ==================================================

dbutils.widgets.text("CATALOG", "vp_vk_mkc20040528")
CATALOG = dbutils.widgets.get("CATALOG")

BRONZE_TABLE = f"{CATALOG}.bronze.chargers"
SILVER_TABLE = f"{CATALOG}.silver.chargers"

In [0]:

from pyspark.sql.functions import col, trim, upper, to_date
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

df = spark.read.table(BRONZE_TABLE)

# Extract fields directly from the existing connector STRUCT
df = df.select(
    upper(trim(col("charger_id"))).alias("charger_id"),
    upper(trim(col("station_id"))).alias("station_id"),
    upper(trim(col("connector.type"))).alias("connector_type"),
    col("connector.max_kw").cast("decimal(6,1)").alias("max_kw"),
    col("is_active").cast("boolean").alias("is_active"),
    to_date(col("commissioned_on"), "yyyy-MM-dd").alias("commissioned_on"),
    to_date(col("snapshot_date"), "yyyy-MM-dd").alias("snapshot_date")
)

# Keep the latest snapshot per charger
window_spec = Window.partitionBy("charger_id").orderBy(
    col("snapshot_date").desc_nulls_last()
)

df = (
    df.withColumn("_rn", row_number().over(window_spec))
      .filter(col("_rn") == 1)
      .drop("_rn")
)

# Create source view for the SQL MERGE
df.createOrReplaceTempView("chargers_source")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS IDENTIFIER(:CATALOG).silver.chargers (
    connector_type STRING,
    charger_id STRING,
    station_id STRING,
    max_kw DECIMAL(6,1),
    is_active BOOLEAN,
    commissioned_on DATE,
    last_changed_snapshot_date DATE
);

In [0]:
%sql

MERGE INTO IDENTIFIER(:CATALOG).silver.chargers AS t
USING (
    SELECT
        connector_type,
        charger_id,
        station_id,
        max_kw,
        is_active,
        commissioned_on,
        snapshot_date
    FROM chargers_source
    WHERE charger_id IS NOT NULL
) AS s

ON t.charger_id = s.charger_id

WHEN MATCHED AND (
       NOT (t.connector_type <=> s.connector_type)
    OR NOT (t.station_id <=> s.station_id)
    OR NOT (t.max_kw <=> s.max_kw)
    OR NOT (t.is_active <=> s.is_active)
    OR NOT (t.commissioned_on <=> s.commissioned_on)
)
THEN UPDATE SET
    t.connector_type = s.connector_type,
    t.station_id = s.station_id,
    t.max_kw = s.max_kw,
    t.is_active = s.is_active,
    t.commissioned_on = s.commissioned_on,
    t.last_changed_snapshot_date = s.snapshot_date

WHEN NOT MATCHED THEN INSERT (
    connector_type,
    charger_id,
    station_id,
    max_kw,
    is_active,
    commissioned_on,
    last_changed_snapshot_date
)
VALUES (
    s.connector_type,
    s.charger_id,
    s.station_id,
    s.max_kw,
    s.is_active,
    s.commissioned_on,
    s.snapshot_date
);


## ENABLE CHANGE DATA FEED BEFORE BATCH 2 LOAD

In [0]:
%sql
ALTER TABLE IDENTIFIER(:CATALOG).silver.chargers
SET TBLPROPERTIES (
    delta.enableChangeDataFeed = true
);